# Autoencoders

_Modern Deep Learning & AI_

**Squeeze the input through a tiny bottleneck, then rebuild it. What survives is the essence.**

An autoencoder is a network that tries to copy its input to its output. But there is a catch in the middle.

     The encoder squeezes the input down to a few numbers, called the code or bottleneck. The decoder expands those few numbers back to a full reconstruction.

---

This notebook is a practice scaffold for the **Autoencoders** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build the autoencoder in three steps: (1) define the encoder/decoder network with a tiny bottleneck, (2) push a synthetic batch through it to get a reconstruction, (3) score the reconstruction with MSE and backprop. The whole point is the **bottleneck** in the middle — the code is far smaller than the input, so the network must keep only what matters.

### Step 1 — Define the encoder and decoder

The `encoder` maps each 5-feature input down to a 2-number **code** (the bottleneck), passing through a 4-unit hidden layer with a ReLU. The `decoder` mirrors that, expanding the 2-number code back up to 5 features. `forward` runs both and returns the reconstruction *and* the code, so we can inspect what got squeezed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Autoencoder(nn.Module):
    def __init__(self, n_in=5, code=2):
        super().__init__()
        # encoder: n_in -> 4 -> code (squeeze down to the bottleneck)
        self.encoder = nn.Sequential(
            nn.Linear(n_in, 4),
            nn.ReLU(),
            nn.Linear(4, code),
        )
        # decoder: code -> 4 -> n_in (expand back to full size)
        self.decoder = nn.Sequential(
            nn.Linear(code, 4),
            nn.ReLU(),
            nn.Linear(4, n_in),
        )

    def forward(self, x):
        z = self.encoder(x)            # tiny code (bottleneck)
        x_hat = self.decoder(z)        # reconstruction from the code
        return x_hat, z                # reconstruction + code

### Step 2 — Push a synthetic batch through the network

We seed the random generator for reproducibility, build the model, and make 8 fake samples with 5 features each. Running them through the model gives back the reconstruction `x_hat` and the 2-number code `z` for each sample.

In [ ]:
torch.manual_seed(0)
model = Autoencoder(n_in=5, code=2)

x = torch.rand(8, 5)               # 8 synthetic samples, 5 features each
x_hat, z = model(x)                # reconstruction + code

### Step 3 — Score the reconstruction and backprop

The training signal is simple: how close is the rebuilt output to the original input? We measure that with **MSE reconstruction loss** and call `backward()`, which sends gradients back through both the decoder and the encoder. The shapes confirm the squeeze: 5 features in, 2 numbers in the code, 5 features back out.

In [ ]:
loss = F.mse_loss(x_hat, x)        # reconstruction loss
loss.backward()                    # gradients flow back through both nets

print("code shape:", z.shape)      # (8, 2) -> squeezed to 2 numbers
print("recon shape:", x_hat.shape) # (8, 5)
print("recon MSE:", round(loss.item(), 4))

## Visualize the data & results

_Encode real handwritten digits to a 2-D code: do the digit classes separate, and which images reconstruct badly?_

A **linear** autoencoder with MSE loss learns the same subspace as PCA, so we use `PCA(2)` as a stand-in for the trained encoder/decoder. We do it in three steps: (1) load and scale the real digits, (2) encode to a 2-D code and decode back, measuring per-image error, (3) plot the code and the worst reconstructions.

### Step 1 — Load and scale the real digit images

We load 1797 real 8x8 handwritten-digit images, flatten each to 64 pixels, and scale intensities into 0..1. `y` keeps the digit label for each image so we can colour the code by class later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# linear autoencoder = PCA(2) on REAL handwritten digits (8x8 images)
digits = load_digits()
X = digits.data / 16.0                  # 1797 images, 64 pixels, scaled 0..1
y = digits.target

### Step 2 — Encode to the 2-D code, decode, and measure error

`fit_transform` plays the **encoder**: it squeezes each 64-pixel image to a 2-number code `Z`. `inverse_transform` plays the **decoder**: it rebuilds the 64 pixels from those 2 numbers. The per-image MSE between original and reconstruction tells us which digits the tiny code captures well and which it loses.

In [ ]:
pca = PCA(n_components=2)
Z = pca.fit_transform(X)                 # 2-D bottleneck code
recon = pca.inverse_transform(Z)         # decode back to 64 pixels
mse = np.mean((X - recon) ** 2, axis=1)  # per-image reconstruction error

### Step 3 — Plot the code and the hardest reconstructions

Left: the first few digit classes plotted in the 2-D code space — do they form separable clusters? Right: a few typical-error images alongside the two hardest-to-reconstruct ones (red), showing where a 2-number bottleneck simply cannot hold enough detail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left — the digits laid out in the 2-D code
class_colors = [(0, "#4ea1ff"), (1, "#7ee787"), (2, "#ffb454")]
for c, col in class_colors:
    idx = np.where(y == c)[0][:20]
    axes[0].scatter(Z[idx, 0], Z[idx, 1], color=col, label="digit %d" % c)
axes[0].set_xlabel("code dim 1")
axes[0].set_ylabel("code dim 2")
axes[0].set_title("digits in the 2-D code")
axes[0].legend()

# Right — typical-error images plus the two hardest to reconstruct
order = np.argsort(mse)               # image indices sorted by error
typical = list(order[200:206])        # six middling-error images
hardest = list(order[-2:])            # two worst-error images
pick = typical + hardest

typical_labels = ["d%d" % y[i] for i in typical]
hardest_labels = ["hard d%d" % y[i] for i in hardest]
lab = typical_labels + hardest_labels
col = ["#7ee787"] * 6 + ["#ff7b72"] * 2

axes[1].bar(lab, mse[pick], color=col)
axes[1].set_title("reconstruction MSE per image")
plt.show()